In [3]:
"""
Hybrid Photonic-Quantum Reservoir Computing for Swaption Pricing
=================================================================
Original implementation combining:
  - Perceval photonic quantum circuits as quantum reservoirs
  - Classical simulation of photonic time-delay reservoir
  - Ridge regression readout layer

Pipeline:
  prices (224) -> StandardScaler -> PCA(5) -> windows(5) ->
  MinMaxScaler [0,pi] ->
  Quantum Ensemble (5x Perceval) || Photonic Reservoir ->
  Concatenate + raw -> Ridge(0.01) -> PCA^-1 -> Scaler^-1 -> prices
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import perceval as pcvl
from merlin import QuantumLayer, ComputationSpace, LexGrouping
from pathlib import Path
from tqdm import tqdm
import aqora_cli
import polars as pl
from aqora_cli.pyarrow import dataset

torch.manual_seed(42)
np.random.seed(42)


# =============================================================================
# HYPERPARAMETERS
# =============================================================================

CFG = {
    # Data
    "n_pca"           : 5,      
    "window"          : 5,      

    # Quantum circuit
    "n_modes"         : 8,      
    "n_photons"       : 3,      
    "encode_modes"    : 5,      
    "n_layers"        : 5,      
    "n_reservoirs"    : 5,      
    "lex_out"         : 10,     

    # Photonic reservoir
    "ph_units"        : 64,     
    "ph_delay"        : 5,      
    "ph_sparsity"     : 0.3,    
    "ph_spectral_r"   : 0.95,   
    "ph_feedback"     : 0.1,    

    # Readout
    "ridge_alpha"     : 0.01,   
}

# Derived
CFG["total_enc"]    = CFG["encode_modes"] * CFG["n_layers"]   # 25
CFG["input_state"]  = [1]*CFG["n_photons"] + [0]*(CFG["n_modes"]-CFG["n_photons"])


# =============================================================================
# PART 1 — PHOTONIC RESERVOIR
# =============================================================================

class PhotonicReservoir:
    """
    Classical simulation of a photonic time-delay reservoir.

    Inspired by silicon-nitride waveguide arrays with Kerr nonlinearity.
    Three contributions at each step:
      (1) Input drive:      W_in  @ x
      (2) Recurrent memory: W_rec @ state
      (3) Delay feedback:   coupling * state[t - delay]

    Weights are random and fixed — only the Ridge layer learns.
    """

    def __init__(self, input_dim, cfg, seed=7):
        rng = np.random.RandomState(seed)
        sz  = cfg["ph_units"]

        # Fixed random input projection
        self.W_in = rng.uniform(-1, 1, (sz, input_dim))

        # Sparse recurrent weight matrix
        W = rng.randn(sz, sz) * (rng.rand(sz, sz) < cfg["ph_sparsity"])
        eigvals = np.linalg.eigvals(W)
        radius  = np.max(np.abs(eigvals))
        self.W_rec = W * (cfg["ph_spectral_r"] / radius) if radius > 0 else W

        self.delay    = cfg["ph_delay"]
        self.feedback = cfg["ph_feedback"]
        self.sz       = sz
        self._reset()

    def _reset(self):
        self.h     = np.zeros(self.sz)
        self.buf   = np.zeros((self.delay, self.sz))  # circular delay buffer

    def _step(self, x):
        """
        One time step.
        Kerr nonlinearity approximated by tanh (captures saturation).
        """
        delayed = self.buf[-1]
        drive   = self.W_in @ x + self.W_rec @ self.h + self.feedback * delayed
        self.h  = np.tanh(drive)
        self.buf = np.roll(self.buf, 1, axis=0)
        self.buf[0] = self.h
        return self.h.copy()

    def run(self, X):
        """
        Process a full sequence X of shape (T, input_dim).
        Returns array of shape (T, ph_units).
        """
        self._reset()
        return np.vstack([self._step(x) for x in X])

    def step_single(self, x):
        """Single step without reset — used during autoregressive inference."""
        return self._step(x)

    def reset(self):
        self._reset()


# =============================================================================
# PART 2 — QUANTUM RESERVOIR
# =============================================================================

def _make_circuit(cfg):
    """
    Build one Perceval photonic circuit.

    Each of the n_layers layers consists of:
      - A random interferometer (beam splitters + phase shifters) — reservoir
      - Phase shifters on the first encode_modes modes — data encoding
    Followed by one final interferometer.
    """
    n     = cfg["n_modes"]
    n_enc = cfg["encode_modes"]
    enc_counter = 0
    circuit = pcvl.Circuit(n)

    for lyr in range(cfg["n_layers"]):
        # Trainable-named but fixed interferometer (reservoir weights)
        circuit.add(0, pcvl.GenericInterferometer(
            n,
            lambda i, l=lyr: (
                pcvl.BS() // pcvl.PS(pcvl.P(f"r{l}_{i}a"))
                         // pcvl.BS() // pcvl.PS(pcvl.P(f"r{l}_{i}b"))
            ),
            shape=pcvl.InterferometerShape.RECTANGLE,
        ))
        # Data encoding phase shifters
        for m in range(n_enc):
            circuit.add(m, pcvl.PS(pcvl.P(f"enc_{enc_counter}")))
            enc_counter += 1

    # Final mixing layer
    circuit.add(0, pcvl.GenericInterferometer(
        n,
        lambda i: (
            pcvl.BS() // pcvl.PS(pcvl.P(f"rf_{i}a"))
                     // pcvl.BS() // pcvl.PS(pcvl.P(f"rf_{i}b"))
        ),
        shape=pcvl.InterferometerShape.RECTANGLE,
    ))

    return circuit, enc_counter


def build_quantum_ensemble(cfg):
    """
    Build cfg["n_reservoirs"] independent quantum reservoirs.
    Each uses a different random seed -> diverse feature extraction.
    Returns a list of nn.Sequential (QuantumLayer + LexGrouping), eval mode.
    """
    ensemble = []
    print(f"  Building {cfg['n_reservoirs']} quantum reservoirs...")

    for idx in range(cfg["n_reservoirs"]):
        torch.manual_seed(1000 + idx * 37)
        circuit, n_enc_params = _make_circuit(cfg)

        core = QuantumLayer(
            input_size          = n_enc_params,
            circuit             = circuit,
            input_state         = cfg["input_state"],
            input_parameters    = ["enc"],
            trainable_parameters= ["r", "rf"],
            computation_space   = ComputationSpace.UNBUNCHED,
            dtype               = torch.float32,
        )

        reservoir = nn.Sequential(core, LexGrouping(core.output_size, cfg["lex_out"]))
        reservoir.eval()
        ensemble.append(reservoir)
        print(f"    [{idx+1}/{cfg['n_reservoirs']}] done")

    return ensemble


def quantum_encode(x, ensemble, cfg):
    """
    Pass one input vector through every quantum reservoir.
    x: 1D numpy array
    Returns: 1D numpy array of size (n_reservoirs * lex_out)
    """
    total = cfg["total_enc"]
    x = x[:total] if len(x) >= total else np.pad(x, (0, total - len(x)))
    xt = torch.tensor(x, dtype=torch.float32).unsqueeze(0)

    parts = []
    with torch.no_grad():
        for res in ensemble:
            parts.append(res(xt).squeeze(0).numpy())

    return np.concatenate(parts)



def load_swaptions():
    """Load swaption data directly from the Quandela aqora dataset."""
    print("  Fetching dataset from aqora...")
    df_raw = pl.scan_pyarrow_dataset(
        dataset("quandela/challenge-swpations", "v0.0.0")
    ).collect()
    df = df_raw.to_pandas()
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
    df = df.sort_values("Date").reset_index(drop=True)
    cols   = [c for c in df.columns if c != "Date"]
    prices = df[cols].values.astype(np.float64)
    print(f"  {len(prices)} days x {len(cols)} swaptions loaded")
    return prices, cols, df["Date"].values


def build_windows(pc_series, window):
    """
    pc_series: (T, n_pca)
    Returns X: (T-window, window*n_pca), y: (T-window, n_pca)
    """
    X, y = [], []
    for t in range(window, len(pc_series)):
        X.append(pc_series[t - window : t].ravel())
        y.append(pc_series[t])
    return np.array(X), np.array(y)



def train_hpqrc(prices, cfg):
    """
    Fit the full HPQRC pipeline on raw swaption price data.

    Returns a model dict containing every fitted object needed for inference.
    """
    # ── Step 1: Preprocessing ────────────────────────────────────────────
    pipeline = tqdm(total=5, desc="Training Pipeline", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}]")

    pipeline.set_postfix_str("Scaler + PCA + Windows")
    std_scaler = StandardScaler().fit(prices)
    scaled     = std_scaler.transform(prices)
    pca        = PCA(n_components=cfg["n_pca"]).fit(scaled)
    pc         = pca.transform(scaled)
    cumvar     = np.cumsum(pca.explained_variance_ratio_)[-1]
    X_raw, y   = build_windows(pc, cfg["window"])
    pipeline.update(1)

    # ── Step 2: Amplitude encoding ───────────────────────────────────────
    pipeline.set_postfix_str("Amplitude encoding [0, pi]")
    amp_scaler = MinMaxScaler(feature_range=(0, np.pi))
    X_enc      = amp_scaler.fit_transform(X_raw)
    pipeline.update(1)

    # ── Step 3: Quantum feature extraction ───────────────────────────────
    pipeline.set_postfix_str("Building quantum reservoirs")
    q_ensemble = build_quantum_ensemble(cfg)

    Q_list = []
    q_loop = tqdm(X_enc, desc="  Quantum features", leave=False)
    for x in q_loop:
        Q_list.append(quantum_encode(x, q_ensemble, cfg))
        q_loop.set_postfix({"features": len(Q_list), "dim": cfg["n_reservoirs"] * cfg["lex_out"]})
    Q = np.array(Q_list)
    pipeline.set_postfix_str(f"Quantum done — shape {Q.shape}")
    pipeline.update(1)

    # ── Step 4: Photonic feature extraction ──────────────────────────────
    pipeline.set_postfix_str("Running photonic reservoir")
    ph_res  = PhotonicReservoir(X_enc.shape[1], cfg)
    P_list  = []
    ph_loop = tqdm(X_enc, desc="  Photonic features", leave=False)
    for x in ph_loop:
        P_list.append(ph_res.step_single(x))
        ph_loop.set_postfix({"features": len(P_list), "dim": cfg["ph_units"]})
    P = np.array(P_list)
    pipeline.set_postfix_str(f"Photonic done — shape {P.shape}")
    pipeline.update(1)

    # ── Step 5: Ridge regression ─────────────────────────────────────────
    pipeline.set_postfix_str("Fitting Ridge regression")
    F     = np.hstack([Q, P, X_raw])   # (n_samples, 139)
    ridge = Ridge(alpha=cfg["ridge_alpha"])
    ridge.fit(F, y)

    y_hat_pca    = ridge.predict(F)
    y_hat_prices = std_scaler.inverse_transform(pca.inverse_transform(y_hat_pca))
    y_true_prices= std_scaler.inverse_transform(pca.inverse_transform(y))
    train_r2     = r2_score(y.ravel(), y_hat_pca.ravel())
    pipeline.set_postfix_str(f"Ridge done — train R2={train_r2:.5f}")
    pipeline.update(1)
    pipeline.close()

    print(f"\nTraining Finished!")
    print(f"  PCA variance captured : {cumvar*100:.1f}%")
    print(f"  Feature vector size   : {F.shape[1]} (Q={Q.shape[1]}, P={P.shape[1]}, raw={X_raw.shape[1]})")
    print(f"  Training R2 (PCA)     : {train_r2:.6f}")

    return {
        "std_scaler"   : std_scaler,
        "pca"          : pca,
        "amp_scaler"   : amp_scaler,
        "q_ensemble"   : q_ensemble,
        "ph_res"       : ph_res,
        "ridge"        : ridge,
        "last_window"  : pc[-cfg["window"]:],
        "diag": {
            "y_true_pca"    : y,
            "y_hat_pca"     : y_hat_pca,
            "y_true_prices" : y_true_prices,
            "y_hat_prices"  : y_hat_prices,
            "pca_var"       : pca.explained_variance_ratio_,
        }
    }



def predict_hpqrc(model, n_days, cfg):
    """
    Autoregressively predict n_days forward.
    Each prediction is fed back as input for the next step.

    Returns: np.array of shape (n_days, 224)
    """
    buffer = list(model["last_window"])
    model["ph_res"].reset()
    preds_pca = []

    for _ in range(n_days):
        x_raw = np.array(buffer[-cfg["window"]:]).ravel()
        x_enc = model["amp_scaler"].transform(x_raw.reshape(1, -1))[0]

        q = quantum_encode(x_enc, model["q_ensemble"], cfg)
        p = model["ph_res"].step_single(x_enc)
        f = np.concatenate([q, p, x_raw]).reshape(1, -1)

        next_pc = model["ridge"].predict(f)[0]
        preds_pca.append(next_pc)
        buffer.append(next_pc)

    preds_pca    = np.array(preds_pca)
    preds_scaled = model["pca"].inverse_transform(preds_pca)
    return model["std_scaler"].inverse_transform(preds_scaled)

import joblib

# Save
def save_model(model, path="hpqrc_results/model.pkl"):
    # Can't pickle torch nn.Sequential directly, so save separately
    joblib.dump({
        "std_scaler"  : model["std_scaler"],
        "pca"         : model["pca"],
        "amp_scaler"  : model["amp_scaler"],
        "ph_res"      : model["ph_res"],
        "ridge"       : model["ridge"],
        "last_window" : model["last_window"],
    }, path)
    # Save quantum ensemble separately
    torch.save(
        [res.state_dict() for res in model["q_ensemble"]],
        "hpqrc_results/quantum_ensemble.pt"
    )
    print(f"  Model saved to {path}")
    print(f"  Quantum ensemble saved to hpqrc_results/quantum_ensemble.pt")


def evaluate_on_test_file(model, test_path, train_prices, cfg):
    """
    Use test.xlsx as ground truth to evaluate model performance.
    Each day is predicted using only real prior days as context —
    no error compounding, true out-of-sample evaluation.
    """
    test_df   = pd.read_excel(test_path)
    feat_cols = [c for c in test_df.columns if c != "Date"]
    test_px   = test_df[feat_cols].values.astype(np.float64)  # (6, 224)
    dates     = test_df["Date"].tolist()

    # Seed buffer with last cfg["window"] real training days
    real_buffer = list(train_prices[-cfg["window"]:])
    model["ph_res"].reset()
    predictions = []

    for i in range(len(test_px)):
        # Build window from real prices only
        window_prices = np.array(real_buffer[-cfg["window"]:])
        scaled        = model["std_scaler"].transform(window_prices)
        pc_window     = model["pca"].transform(scaled)

        # Encode + extract features
        x_raw = pc_window.ravel()
        x_enc = model["amp_scaler"].transform(x_raw.reshape(1, -1))[0]
        q     = quantum_encode(x_enc, model["q_ensemble"], cfg)
        p     = model["ph_res"].step_single(x_enc)
        f     = np.concatenate([q, p, x_raw]).reshape(1, -1)

        # Predict and inverse transform
        pred_pca    = model["ridge"].predict(f)[0]
        pred_scaled = model["pca"].inverse_transform(pred_pca.reshape(1, -1))
        pred_prices = model["std_scaler"].inverse_transform(pred_scaled)[0]
        pred_prices = np.clip(pred_prices, 0.001, None)
        predictions.append(pred_prices)

        # Feed REAL value back — not the prediction
        real_buffer.append(test_px[i])

    predictions = np.array(predictions)  # (6, 224)

    # ── Metrics ──────────────────────────────────────────────────────────
    print(f"\n{'Date':<14}{'MSE':>14}{'MAE':>12}{'RelErr%':>10}")
    print("-" * 52)
    for i, date in enumerate(dates):
        mse = mean_squared_error(test_px[i], predictions[i])
        mae = np.mean(np.abs(test_px[i] - predictions[i]))
        rel = 100 * np.mean(
            np.abs(test_px[i] - predictions[i]) / (np.abs(test_px[i]) + 1e-10)
        )
        print(f"{date:<14}{mse:>14.2e}{mae:>12.6f}{rel:>9.4f}%")

    overall_mse = mean_squared_error(test_px.ravel(), predictions.ravel())
    overall_r2  = r2_score(test_px.ravel(), predictions.ravel())
    print(f"\nOverall MSE : {overall_mse:.2e}")
    print(f"Overall R2  : {overall_r2:.6f}")

    # ── Save ─────────────────────────────────────────────────────────────
    out = Path("hpqrc_results")
    out.mkdir(parents=True, exist_ok=True)
    for label, data in [("test_actual", test_px), ("test_predicted", predictions)]:
        pd.DataFrame(data, columns=feat_cols, index=dates
                     ).to_csv(out / f"{label}.csv")
    print(f"\nSaved test_actual.csv and test_predicted.csv to hpqrc_results/")

    return predictions, test_px


# =============================================================================
# PART 6 — EVALUATION & PLOTS
# =============================================================================

def print_metrics(actual, predicted, train_days, pred_days):
    print(f"\n{'Day':<8}{'MSE':>14}{'MAE':>12}{'RelErr%':>10}")
    print("-" * 46)
    for d in range(pred_days):
        mse = mean_squared_error(actual[d], predicted[d])
        mae = np.mean(np.abs(actual[d] - predicted[d]))
        rel = 100 * np.mean(
            np.abs(actual[d] - predicted[d]) / (np.abs(actual[d]) + 1e-10)
        )
        print(f"{train_days+d+1:<8}{mse:>14.2e}{mae:>12.6f}{rel:>9.4f}%")
    overall_mse = mean_squared_error(actual.ravel(), predicted.ravel())
    overall_r2  = r2_score(actual.ravel(), predicted.ravel())
    print(f"\nOverall MSE : {overall_mse:.2e}")
    print(f"Overall R2  : {overall_r2:.6f}")


def save_plots(model, actual, predicted, feat_cols, train_days, pred_days, out):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    diag = model["diag"]

    # Training fit — PCA space
    n = diag["y_true_pca"].shape[1]
    fig, axes = plt.subplots(n, 1, figsize=(14, 3*n), sharex=True)
    axes = [axes] if n == 1 else axes
    for i in range(n):
        r2 = r2_score(diag["y_true_pca"][:, i], diag["y_hat_pca"][:, i])
        axes[i].plot(diag["y_true_pca"][:, i], lw=0.8, label="Actual")
        axes[i].plot(diag["y_hat_pca"][:, i],  lw=0.8, label="Predicted", alpha=0.7)
        axes[i].set_title(f"PC{i+1} — train R2={r2:.4f}")
        axes[i].legend(fontsize=8)
    fig.suptitle("Training Fit — PCA Space")
    fig.tight_layout()
    fig.savefig(out / "train_pca.png", dpi=150)
    plt.close(fig)

    # Scatter: predicted vs actual
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(actual.ravel(), predicted.ravel(), s=3, alpha=0.4)
    lo = min(actual.min(), predicted.min())
    hi = max(actual.max(), predicted.max())
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.5, label="Perfect")
    r2  = r2_score(actual.ravel(), predicted.ravel())
    mse = mean_squared_error(actual.ravel(), predicted.ravel())
    ax.set_title(f"Inference  R2={r2:.4f}  MSE={mse:.2e}")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "inference_scatter.png", dpi=150)
    plt.close(fig)

    # Error heatmap
    err  = predicted - actual
    vabs = np.max(np.abs(err))
    fig, ax = plt.subplots(figsize=(16, 3))
    im = ax.imshow(err, aspect="auto", cmap="RdBu_r", vmin=-vabs, vmax=vabs)
    ax.set_yticks(range(pred_days))
    ax.set_yticklabels([f"Day {train_days+d+1}" for d in range(pred_days)])
    ax.set_xlabel("Swaption index (224)")
    ax.set_title("Prediction Error Heatmap")
    plt.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    fig.savefig(out / "error_heatmap.png", dpi=150)
    plt.close(fig)

    # PCA variance
    var    = diag["pca_var"]
    cumvar = np.cumsum(var)
    x      = np.arange(1, len(var)+1)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x, var*100, color="#3498db", alpha=0.7)
    ax.plot(x, cumvar*100, "o-", color="#e74c3c", lw=2, label="Cumulative")
    for i, cv in enumerate(cumvar):
        ax.text(x[i], cv*100+1.5, f"{cv*100:.1f}%", ha="center", fontsize=9)
    ax.set_ylim(0, 115)
    ax.set_xlabel("PC")
    ax.set_ylabel("Variance (%)")
    ax.set_title("PCA Variance Explained")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "pca_variance.png", dpi=150)
    plt.close(fig)

    print(f"  Plots saved to {out}/")


if __name__ == "__main__":

    print("=" * 60)
    print("  HPQRC  Swaption Volatility Surface Forecasting")
    print("=" * 60)

    print("\n>>> Loading data")
    prices, feat_cols, dates = load_swaptions()

    TRAIN_DAYS = 450
    PRED_DAYS  = 6
    prices_train = prices[:TRAIN_DAYS]
    prices_test  = prices[TRAIN_DAYS : TRAIN_DAYS + PRED_DAYS]

    print(f"\n>>> Training on {TRAIN_DAYS} days")
    model = train_hpqrc(prices_train, CFG)

    print(f"\n>>> Predicting days {TRAIN_DAYS+1} to {TRAIN_DAYS+PRED_DAYS}")
    predicted = predict_hpqrc(model, PRED_DAYS, CFG)
    print_metrics(prices_test, predicted, TRAIN_DAYS, PRED_DAYS)
    
    out = Path("hpqrc_results")
    out.mkdir(parents=True, exist_ok=True)
    print("\n>>> Saving model")
    save_model(model)
    for label, data in [("actual", prices_test), ("predicted", predicted)]:
        pd.DataFrame(data, columns=feat_cols,
                     index=[f"Day {TRAIN_DAYS+d+1}" for d in range(PRED_DAYS)]
                     ).to_csv(out / f"{label}.csv")
    save_plots(model, prices_test, predicted, feat_cols,
               TRAIN_DAYS, PRED_DAYS, out)

    # For test.xlsx evaluation — pass your local path here
    test_path = Path(r"C:\Users\ziyad\Downloads\test.xlsx")
    if test_path.exists():
        print("\n>>> Evaluating on test.xlsx")
        evaluate_on_test_file(model, test_path, prices_train, CFG)

    print("\n>>> Done.")

  HPQRC  Swaption Volatility Surface Forecasting

>>> Loading data
  Fetching dataset from aqora...
  494 days x 224 swaptions loaded

>>> Training on 450 days


  Building 5 quantum reservoirs...
    [1/5] done
    [2/5] done
    [3/5] done
    [4/5] done
    [5/5] done



Training Finished!
  PCA variance captured : 100.0%
  Feature vector size   : 139 (Q=50, P=64, raw=25)
  Training R2 (PCA)     : 0.971915

>>> Predicting days 451 to 456

Day                MSE         MAE   RelErr%
----------------------------------------------
451           3.85e-06    0.001512   1.0472%
452           3.43e-06    0.001547   0.7866%
453           5.50e-06    0.001967   1.2526%
454           5.95e-05    0.007229   3.9557%
455           8.79e-06    0.002317   1.7077%
456           3.49e-05    0.004987   3.4680%

Overall MSE : 1.93e-05
Overall R2  : 0.998003

>>> Saving model
  Model saved to hpqrc_results/model.pkl
  Quantum ensemble saved to hpqrc_results/quantum_ensemble.pt
  Plots saved to hpqrc_results/

>>> Evaluating on test.xlsx

Date                     MSE         MAE   RelErr%
----------------------------------------------------
24/12/2051          1.79e-02    0.108157  94.6562%
26/12/2051          4.23e-02    0.181840  86.7075%
27/12/2051          4.43e-02  